In [ ]:
import time
import numpy as np
import torch
import h5py
# import matplotlib.pyplot as plt
from mlrg.hmc import NewHMCSampler
from rgflow import RGPartition, RealNVP
from rgflow import CNN
# from functorch import make_functional_with_buffers, make_functional
# from torchviz import make_dot

In [ ]:
class Phi4Model(torch.nn.Module):
    ''' phi4 model energy
        E = (1/2) sum_<ij> |x_i-x_j|^2 + (r/2) sum_i |x_i|^2 + (u/4!) sum_i |x_i|^4

        Parameters:
            r :: real - (initial) value of r
            u :: real - (initial) value of u '''
    def __init__(self):
        super().__init__()


    def extra_repr(self):
        return f'r={self.r.item()}, u={self.u.item()}'

    def return_param(self):
        return self.r, self.u

    def clone(self):
        mdl = type(self)()
        mdl.load_state_dict(self.state_dict())
        return mdl

    def forward(self, x, p_uv):
        energy = 0.
        for axis in range(2, x.dim()):
            dx2 = (x.roll(1,axis) - x).square().sum(1)
            energy = energy + dx2 / 2
        x2 = x.square().sum(1)
        energy = energy + p_uv[0] * x2 / 2 + p_uv[1] * x2.square() / 4
        energy = energy.view(energy.shape[:1]+(-1,)).sum(-1)
        return energy

In [ ]:
class Phi4Model_prediction(torch.nn.Module):
    ''' phi4 model energy
        E = (1/2) sum_<ij> |x_i-x_j|^2 + (r/2) sum_i |x_i|^2 + (u/4!) sum_i |x_i|^4

        Parameters:
            r :: real - (initial) value of r
            u :: real - (initial) value of u '''
    def __init__(self, r, u):
        super().__init__()
        self.r = torch.nn.Parameter(r)
        self.u = torch.nn.Parameter(u)

    def extra_repr(self):
        return f'r={self.r.item()}, u={self.u.item()}'

    def return_param(self):
        return self.r, self.u

    def return_increment(self, p_uv):
        return self.r, (p_uv[1] + self.u) - p_uv[1]

    def clone(self):
        mdl = type(self)()
        mdl.load_state_dict(self.state_dict())
        return mdl

    def forward(self, x, p_uv):
        energy = 0.
        for axis in range(2, x.dim()):
            dx2 = (x.roll(1,axis) - x).square().sum(1)
            energy = energy + dx2 / 2
        x2 = x.square().sum(1)
        energy = energy + (p_uv[0] + self.r) * x2 / 2 + (p_uv[1] + self.u) * x2.square() / 4
        energy = energy.view(energy.shape[:1]+(-1,)).sum(-1)
        return energy

In [ ]:
class Real_NVP_learner(torch.nn.Module):
    def __init__(self, ir_model, mask, device, uv_shape, dim, layers, base_dist='Normal', **kwargs):
        super().__init__()
        self.uv_shape = uv_shape
        self.d = len(uv_shape)
        self.ir_model = ir_model.requires_grad_(True).to(device)
        self.nets = lambda: torch.nn.Sequential(CNN(self.d, dim, hdims=[8], activation='LeakyReLU')).to(device)
        self.nett = lambda: torch.nn.Sequential(CNN(self.d, dim, hdims=[8], activation='LeakyReLU')).to(device)
        self.mask = mask * layers
        self.NVPlayer = RealNVP(self.nets, self.nett, self.mask, uv_shape, **kwargs).requires_grad_(True).to(device)
        # self.HyperNet = Dynamic(self.NVPlayer, device, [128, 256, 512, 1024, 2048]).requires_grad_(True)
        self.partitioner = RGPartition(uv_shape, **kwargs).to(device)
        ir_shape = self.partitioner.out_shape
        self.ir_sampler = NewHMCSampler(self.ir_model, (1, dim)+torch.Size(ir_shape))
        self.base_dist = getattr(torch.distributions, base_dist)(0., 1.)
        self.uv_model = Phi4Model().to(device)


    def latent_score_loss(self, params_uv, samples, device, **kwargs):
        x_ir = self.ir_sampler.sample(params_uv, device, samples=samples, **kwargs)
        z = self.base_dist.rsample(x_ir.shape[:2]+self.partitioner.res_shape).to(device)
        x_ir.requires_grad = True
        z.requires_grad = True
        Z = self.partitioner.merge(x_ir, z)
        # def compute(Z, **kwargs):
        #     Z = Z.view(Z.shape[:2]+tuple(self.uv_shape))
        #     x_1, x_2 = self.HyperNet(params_uv, Z)
        #     x_1 = torch.flatten(x_1, start_dim=2)
        #     return x_1.sum(dim=0), x_2.sum(dim=0)

        x_uv, logJ = self.NVPlayer( Z)
        log_p_uv = - self.uv_model(x_uv, params_uv)
        log_p_ir = - self.ir_model(x_ir, params_uv)
        # d_uv_z = torch.autograd.grad(outputs=log_p_uv, inputs=Z, grad_outputs=torch.ones_like(log_p_uv), create_graph=True)[0]
        d_uv_z = torch.autograd.grad(outputs=log_p_uv.sum(), inputs=Z, create_graph=True)[0]
        # d_ir_z = torch.autograd.grad(outputs=log_p_ir, inputs=x_ir, grad_outputs=torch.ones_like(log_p_ir), create_graph=True)[0]
        d_ir_z = torch.autograd.grad(outputs=log_p_ir.sum(), inputs=x_ir, create_graph=True)[0]
        # d_logJ_z = torch.autograd.grad(outputs=logJ, inputs=Z, grad_outputs=torch.ones_like(logJ), create_graph=True)[0]
        d_logJ_z = torch.autograd.grad(outputs=logJ.sum(), inputs=Z, create_graph=True)[0]
        d_z = - z
        d_Z = self.partitioner.merge(d_ir_z, d_z)
        Loss = torch.linalg.norm(d_Z - d_logJ_z- d_uv_z, dim=(2, 3)) ** 2
        Loss = Loss.mean()
        return Loss

In [ ]:
device = 'cuda'
rlist = [1]
ulist = [0.1]
increment = torch.Tensor([0, 0])
masks = [RGPartition([6, 6]).mask.to(device), ~RGPartition([6, 6]).mask.to(device)]
flow = Real_NVP_learner(Phi4Model_prediction(increment[0], increment[1]).to(device), masks, device, [6, 6], 1, 3)
print(sum(p.numel() for p in flow.parameters() if p.requires_grad))
print(sum(p.numel() for p in flow.NVPlayer.parameters() if p.requires_grad))

In [ ]:
p_uv_r = -0.07
p_uv_u = 0.04
p_uv = torch.Tensor([p_uv_r, p_uv_u]).to(device).requires_grad_(True)
# PATH = 'critical_line_3x3/NVP_score_matching_phi_four_phase_2D_prediction_net_-0.05_0.04_nvp_run_1.pt'
# PATH = 'saddle_3x3/NVP_score_matching_phi_four_phase_2D_prediction_net_-2.2_2.2_nvp_run_1.pt'
# checkpoint = torch.load(PATH)
# flow.NVPlayer.load_state_dict(checkpoint['model_state_dict'])
optimizer = torch.optim.Adam(flow.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5000, gamma=0.8)
r = np.zeros((1, 40001))
u = np.zeros((1, 40001))
L = np.zeros((1, 40001))
eps1 = 0.5
eps2 = 1.0
T = time.time()

for param in flow.NVPlayer.parameters():
    param.requires_grad_(True)

for i in range(100):
    _ = flow.ir_sampler.sample(p_uv, device, samples=1000)

for t in range(40001):
    # r0 = 2 * eps1 * torch.rand(1) + (-1.0 - eps1)
    # u0 = 2 * eps2 * (1 - torch.rand(1)) + (6.0 - eps2)
    # if r0 < 0:
    #     u0 = (0.5 - 3/5 * r0 ** 2) * (1 - torch.rand(1)) + (3/5 * r0 ** 2)
    # else:
    #     u0 = 2 * eps2 * (1 - torch.rand(1)) + (6.0 - eps2)

    # p_uv = torch.cat((r0, u0)).to(device)
    loss = flow.latent_score_loss(p_uv, 1000, device)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    drt, dut = flow.ir_model.return_increment(p_uv)
    rt = drt.cpu().detach().numpy() + p_uv_r#r0.numpy()
    ut = dut.cpu().detach().numpy() + p_uv_u#u0.numpy()
    r[0, t] = rt
    u[0, t] = ut
    L[0, t] = loss
    if t % 100 == 0:
        print('iter %s:' % t, 'loss = %f' % loss, 'r = %f' % rt, 'u = %f' % ut)
print(time.time()-T)

In [ ]:
PATH = 'critical_line_3x3/NVP_score_matching_phi_four_phase_2D_prediction_net_' + str(p_uv_r) + '_' + str(p_uv_u) + '_nvp_run_1.pt'
torch.save({
    'model_state_dict': flow.NVPlayer.state_dict(),
    # 'optimizer_state_dict': optimizer.state_dict(),
    # 'loss': loss,
    }, PATH)

In [ ]:
with h5py.File('critical_line_3x3/training_data_NVP_score_matching_phi_four_2D_prediction_net_' + str(p_uv_r) + '_' + str(p_uv_u) + '_run_1.h5', 'w') as hf:
    hf.create_dataset("r",  data=r)
    hf.create_dataset("u",  data=u)
    hf.create_dataset("L",  data=L)